<a href="https://colab.research.google.com/github/dpthinh10dona/StrongPasswordGenerator/blob/main/StrongPasswordGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from google.colab import userdata
from datasets import Dataset
!pip install -q zxcvbn
import zxcvbn
!pip install -q tqdm

df = pd.read_csv("dataset.csv", nrows=200000)
df.drop(columns="strong", inplace=True)
df.columns = ["password"]
df["password"] = df["password"].astype(str)

df["length"] = df["password"].str.len()
df["lowercase"] = df["password"].str.contains(r'[a-z]').astype(int)
df["uppercase"] = df["password"].str.contains(r'[A-Z]').astype(int)
df["digit"] = df["password"].str.contains(r'\d').astype(int)

df["special"] = df["password"].str.contains(r'[^a-zA-Z0-9]').astype(int)

from tqdm import tqdm
tqdm.pandas()

df["strength"] = df["password"].progress_apply(
    lambda pw: zxcvbn.zxcvbn(pw)["score"]
)
df.to_csv("dataset_processed.csv", index=False)
print(df.head())

In [ ]:
import os
from google.colab import userdata

# ============================================================
# 1. SETUP KAGGLE
# ============================================================
os.environ['KAGGLE_USERNAME'] = "dpthinh10dona"
os.environ['KAGGLE_KEY'] = userdata.get('KaggleKey')

!pip install -q torch pandas scikit-learn tqdm

# ============================================================
# FULL PIPELINE: Password AI System (Fixed)
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from tqdm.auto import tqdm
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Đang chạy trên thiết bị: {device.upper()}")

# ============================================================
# 2. LOAD & CLEAN DATA
# ============================================================
print("⏳ Đang tải dữ liệu...")
df = pd.read_csv('dataset_processed.csv')
df.columns = [col.lower().strip() for col in df.columns]
df = df.dropna(subset=['password', 'strength'])
df['password'] = df['password'].astype(str)

# BUG FIX #4: Kiểm tra và xử lý strength linh hoạt
# (CSV có thể chứa string labels hoặc số nguyên)
print(f"📋 Các giá trị strength gốc: {df['strength'].unique()[:10]}")

if df['strength'].dtype == object:
    strength_mapping = {
        'Too guessable':      0,
        'Very guessable':     1,
        'Somewhat guessable': 2,
        'Safely unguessable': 3,
        'Very unguessable':   4,
    }
    df['strength'] = df['strength'].map(strength_mapping)
    unmapped = df['strength'].isna().sum()
    if unmapped > 0:
        print(f"⚠️ Có {unmapped} dòng không map được strength → bị drop.")
    df = df.dropna(subset=['strength'])

df['strength'] = df['strength'].astype(int)
print(f"✅ Phân phối strength sau xử lý:\n{df['strength'].value_counts().sort_index()}")

# BUG FIX #2: Tách df_rf và df_tf đúng cách
df_rf = df.sample(50000, random_state=42) if len(df) > 50000 else df.copy()
df_tf = df.head(10000).copy()

# ============================================================
# 3. TRAIN STRENGTH CLASSIFIER (Random Forest)
# ============================================================
features = ["length", "lowercase", "uppercase", "digit", "special"]

# Kiểm tra các cột feature có tồn tại không
missing_cols = [f for f in features if f not in df_rf.columns]
if missing_cols:
    raise ValueError(f"❌ Thiếu các cột feature: {missing_cols}. Kiểm tra lại CSV!")

X = df_rf[features]   # BUG FIX #2: dùng df_rf thay vì df
y = df_rf["strength"] # BUG FIX #2: dùng df_rf thay vì df

clf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
clf.fit(X, y)
print("✅ Mô hình RandomForest đã train xong!")

# ============================================================
# 4. PREPARE TRANSFORMER DATASET
# ============================================================
POS_EMB_MAX = 256  # BUG FIX #1: Tăng lên để prompt + generated không bị tràn

def extract_flags(pw):
    return {
        "LEN":   len(pw),
        "LOWER": int(any(c.islower() for c in pw)),
        "UPPER": int(any(c.isupper() for c in pw)),
        "NUM":   int(any(c.isdigit() for c in pw)),
        "SPEC":  int(any(c in "!@#$%^&*" for c in pw)),
    }

lines = []
for pw in df_tf['password']:
    f = extract_flags(pw)
    line = f"<LEN={f['LEN']}><LOWER={f['LOWER']}><UPPER={f['UPPER']}><NUM={f['NUM']}><SPEC={f['SPEC']}>:{pw}"
    lines.append(line)

all_text = ''.join(lines)
chars = sorted(list(set(all_text)))
char2idx = {c: i + 1 for i, c in enumerate(chars)}
char2idx['<PAD>'] = 0
idx2char = {i: c for c, i in char2idx.items()}

vocab_size = len(char2idx)
max_len = 100

class PasswordDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def encode(self, text):
        ids = [char2idx.get(c, 0) for c in text]
        ids = ids[:max_len]
        ids += [0] * (max_len - len(ids))
        return torch.tensor(ids)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        x = self.encode(text[:-1])
        y = self.encode(text[1:])
        return x, y

dataset = PasswordDataset(lines)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# ============================================================
# 5. TRANSFORMER MODEL
# ============================================================
class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, pos_emb_max=POS_EMB_MAX):
        super().__init__()
        d_model = 128
        self.pos_emb_max = pos_emb_max
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(1, pos_emb_max, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)

        # BUG FIX #1: Truncate nếu seq_len vượt quá pos_embedding
        if seq_len > self.pos_emb_max:
            x = x[:, :self.pos_emb_max]
            seq_len = self.pos_emb_max

        x_emb = self.embedding(x) + self.pos_embedding[:, :seq_len, :]

        # BUG FIX #3: Thêm is_causal=True cho PyTorch >= 2.0
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x_emb.device)
        x_out = self.transformer(x_emb, mask=mask, is_causal=True)

        return self.fc(x_out)

# ============================================================
# 6. TRAIN GENERATOR
# ============================================================
print("⏳ Đang huấn luyện AI tạo mật khẩu (Transformer)...")
model = MiniTransformer(vocab_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)

EPOCHS = 5
start_time = time.time()

for epoch in range(EPOCHS):
    total_loss = 0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)

    for x, y in progress_bar:
        x, y = x.to(device), y.to(device)

        out = model(x)
        out = out.reshape(-1, vocab_size)
        y_flat = y.reshape(-1)

        loss = criterion(out, y_flat)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_loss = total_loss / len(dataloader)
    print(f"👉 Epoch {epoch+1} xong — Loss trung bình: {avg_loss:.4f}")

total_mins = (time.time() - start_time) / 60
print(f"✅ Transformer đã train xong! Tổng thời gian: {total_mins:.2f} phút.")

# ============================================================
# 7. GENERATOR & PIPELINE FUNCTIONS
# ============================================================
def generate(model, start_text, max_gen=30, temperature=0.8):
    """
    Sinh ký tự mới từ start_text.
    BUG FIX #1: Luôn truncate input_ids để không vượt quá POS_EMB_MAX.
    """
    model.eval()
    input_ids = [char2idx.get(c, 0) for c in start_text]

    for _ in range(max_gen):
        # Truncate từ đuôi để giữ lại ngữ cảnh gần nhất
        truncated = input_ids[-(POS_EMB_MAX - 1):] if len(input_ids) >= POS_EMB_MAX else input_ids
        x_input = torch.tensor([truncated]).to(device)

        with torch.no_grad():
            out = model(x_input)

        probs = torch.softmax(out[0, -1] / temperature, dim=0).cpu()
        next_id = torch.multinomial(probs, 1).item()

        if next_id == 0:
            break
        input_ids.append(next_id)

    result = ''.join([idx2char.get(i, '') for i in input_ids])
    if ":" in result:
        return result.split(":")[-1].strip()
    return result


def build_prompt(criteria, base_pw):
    return (
        f"<LEN={criteria['LEN']}>"
        f"<LOWER={criteria['LOWER']}>"
        f"<UPPER={criteria['UPPER']}>"
        f"<NUM={criteria['NUM']}>"
        f"<SPEC={criteria['SPEC']}>:{base_pw}"
    )


def upgrade_weak_password(weak_pw, criteria, min_strength=2, max_attempts=100, chars_to_add=4):
    prompt = build_prompt(criteria, weak_pw)
    best_pw = weak_pw
    best_score = 0

    for _ in range(max_attempts):
        enhanced_pw = generate(model, prompt, max_gen=chars_to_add)

        if not enhanced_pw or enhanced_pw == weak_pw:
            continue

        f = extract_flags(enhanced_pw)
        features_input = [[f['LEN'], f['LOWER'], f['UPPER'], f['NUM'], f['SPEC']]]
        score = int(clf.predict(features_input)[0])

        if score >= min_strength:
            return enhanced_pw, score

        if score > best_score:
            best_score = score
            best_pw = enhanced_pw

    return best_pw, best_score

# ============================================================
# 8. TEST SYSTEM
# ============================================================
STRENGTH_LABELS = {
    0: "❌ Rất yếu",
    1: "🟠 Yếu",
    2: "🟡 Trung bình",
    3: "🟢 Khá mạnh",
    4: "💪 Rất mạnh",
}

import pickle
import torch

# ── Save Transformer ──────────────────────────────────────
torch.save({
    'model_state': model.state_dict(),
    'vocab_size':  vocab_size,
    'char2idx':    char2idx,
    'idx2char':    idx2char,
}, 'transformer.pt')

# ── Save Random Forest ────────────────────────────────────
with open('rf_classifier.pkl', 'wb') as f:
    pickle.dump(clf, f)

print("✅ Đã lưu: transformer.pt + rf_classifier.pkl")

print("\n" + "="*50)
print("🔥  HỆ THỐNG NÂNG CẤP MẬT KHẨU BẰNG AI  🔥")
print("="*50)

weak_password = input("👉 Nhập mật khẩu yếu: ").strip()
target_len    = int(input("👉 Số ký tự mong muốn: ").strip())

if target_len <= len(weak_password):
    keep_len     = max(1, target_len - 3)
    weak_password = weak_password[:keep_len]
    chars_to_add = target_len - len(weak_password)
    print(f"\n⚠️  Pass mới ngắn hơn/bằng pass cũ → cắt còn: '{weak_password}'")
else:
    chars_to_add = target_len - len(weak_password)

criteria = {"LEN": target_len, "LOWER": 1, "UPPER": 1, "NUM": 1, "SPEC": 1}

enhanced_pw, score = upgrade_weak_password(
    weak_password,
    criteria,
    min_strength=2,
    chars_to_add=chars_to_add,
)

# Đánh giá pass gốc để so sánh
f_orig = extract_flags(weak_password)
orig_score = int(clf.predict([[f_orig['LEN'], f_orig['LOWER'], f_orig['UPPER'],
                                f_orig['NUM'], f_orig['SPEC']]])[0])

print("\n" + "="*50)
print(f"  ❌ Mật khẩu gốc     : {weak_password}")
print(f"  📊 Độ mạnh gốc      : {orig_score} — {STRENGTH_LABELS.get(orig_score)}")
print(f"  ✅ Mật khẩu nâng cấp: {enhanced_pw}")
print(f"  📊 Độ mạnh mới      : {score} — {STRENGTH_LABELS.get(score)}")
print("="*50)